### cargar los datos

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

listings_clean = pd.read_csv(
    PROCESSED_DIR / "listings_clean.csv"
)

listings_market = pd.read_csv(
    PROCESSED_DIR / "listings_market_eligible.csv"
)

zones = pd.read_csv(
    PROCESSED_DIR / "zones_master.csv"
)

print("Anuncios limpios conservados:", len(listings_clean))
print("Anuncios utilizables por barrio:", len(listings_market))
print("Barrios oficiales:", len(zones))

assert len(listings_clean) == 11826
assert len(listings_market) == 11058
assert len(zones) == 131

Anuncios limpios conservados: 11826
Anuncios utilizables por barrio: 11058
Barrios oficiales: 131


### calcular la cobertura por barrio

In [4]:
total_by_zone = (
    listings_clean.loc[
        listings_clean["zone_key_final"].notna()
    ]
    .groupby("zone_key_final")
    .size()
    .rename("total_listings")
)

eligible_by_zone = (
    listings_market
    .groupby("zone_key_final")
    .size()
    .rename("eligible_listings")
)

coverage_by_zone = zones[
    [
        "zone_key",
        "neighborhood_name",
        "district_name",
    ]
].copy()

coverage_by_zone = coverage_by_zone.merge(
    total_by_zone,
    left_on="zone_key",
    right_index=True,
    how="left",
)

coverage_by_zone = coverage_by_zone.merge(
    eligible_by_zone,
    left_on="zone_key",
    right_index=True,
    how="left",
)

coverage_by_zone[
    ["total_listings", "eligible_listings"]
] = (
    coverage_by_zone[
        ["total_listings", "eligible_listings"]
    ]
    .fillna(0)
    .astype(int)
)

coverage_by_zone["eligible_percentage"] = np.where(
    coverage_by_zone["total_listings"].gt(0),
    (
        coverage_by_zone["eligible_listings"]
        / coverage_by_zone["total_listings"]
        * 100
    ),
    np.nan,
)

print(
    "Barrios con algún anuncio:",
    coverage_by_zone["total_listings"].gt(0).sum(),
)

print(
    "Barrios con anuncios utilizables:",
    coverage_by_zone["eligible_listings"].gt(0).sum(),
)

print(
    "Barrios sin anuncios utilizables:",
    coverage_by_zone["eligible_listings"].eq(0).sum(),
)

display(
    coverage_by_zone["eligible_listings"].describe()
)

Barrios con algún anuncio: 128
Barrios con anuncios utilizables: 128
Barrios sin anuncios utilizables: 3


count    131.000000
mean      84.412214
std       96.571526
min        0.000000
25%       27.500000
50%       60.000000
75%       96.500000
max      600.000000
Name: eligible_listings, dtype: float64

### Identificar los barrios sin datos y con menor cobertura

In [5]:
print("Barrios sin anuncios utilizables:")

display(
    coverage_by_zone.loc[
        coverage_by_zone["eligible_listings"].eq(0),
        [
            "zone_key",
            "neighborhood_name",
            "district_name",
            "total_listings",
            "eligible_listings",
        ],
    ]
)

print("Barrios con menor número de anuncios utilizables:")

display(
    coverage_by_zone[
        [
            "zone_key",
            "neighborhood_name",
            "district_name",
            "total_listings",
            "eligible_listings",
            "eligible_percentage",
        ]
    ]
    .sort_values(
        [
            "eligible_listings",
            "district_name",
            "neighborhood_name",
        ]
    )
    .head(30)
)

Barrios sin anuncios utilizables:


,zone_key,neighborhood_name,district_name,total_listings,eligible_listings
12,MAD-027,Atocha,Arganzuela,0,0
115,MAD-192,Valdebernardo,Vicálvaro,0,0
116,MAD-193,Valderrivas,Vicálvaro,0,0


Barrios con menor número de anuncios utilizables:


,zone_key,neighborhood_name,district_name,total_listings,eligible_listings,eligible_percentage
12,MAD-027,Atocha,Arganzuela,0,0,NaN
115,MAD-192,Valdebernardo,Vicálvaro,0,0,NaN
116,MAD-193,Valderrivas,Vicálvaro,0,0,NaN
86,MAD-142,Horcajo,Moratalaz,2,2,100.000000
127,MAD-212,Aeropuerto,Barajas,4,3,75.000000
85,MAD-141,Pavones,Moratalaz,5,4,80.000000
43,MAD-081,El Pardo,Fuencarral - El Pardo,5,5,100.000000
98,MAD-158,Atalaya,Ciudad Lineal,8,8,100.000000
50,MAD-088,El Goloso,Fuencarral - El Pardo,12,12,100.000000
104,MAD-165,Apóstol Santiago,Hortaleza,12,12,100.000000


### Asignar un nivel de fiabilidad

In [6]:
coverage_by_zone["reliability_level"] = np.select(
    [
        coverage_by_zone["eligible_listings"].eq(0),
        coverage_by_zone["eligible_listings"].between(
            1,
            29,
        ),
        coverage_by_zone["eligible_listings"].between(
            30,
            59,
        ),
        coverage_by_zone["eligible_listings"].ge(60),
    ],
    [
        "no_data",
        "low",
        "medium",
        "high",
    ],
    default="review",
)

print("Niveles de fiabilidad:")

display(
    coverage_by_zone["reliability_level"]
    .value_counts()
)

Niveles de fiabilidad:


reliability_level
high       66
low        32
medium     30
no_data     3
Name: count, dtype: int64

### Crear las estadísticas inmobiliarias por barrio

In [7]:
zone_group = listings_market.groupby(
    "zone_key_final"
)

neighborhood_statistics = zone_group.agg(
    num_listings=(
        "Enlace",
        "nunique",
    ),

    price_eur_mean=(
        "price_eur",
        "mean",
    ),
    price_eur_median=(
        "price_eur",
        "median",
    ),

    price_m2_mean=(
        "price_per_m2",
        "mean",
    ),
    price_m2_median=(
        "price_per_m2",
        "median",
    ),
    price_m2_std=(
        "price_per_m2",
        "std",
    ),

    area_m2_mean=(
        "area_m2",
        "mean",
    ),
    area_m2_median=(
        "area_m2",
        "median",
    ),

    rooms_mean=(
        "rooms_clean",
        "mean",
    ),
    rooms_median=(
        "rooms_clean",
        "median",
    ),
    rooms_known_count=(
        "rooms_clean",
        "count",
    ),

    bathrooms_mean=(
        "bathrooms_clean",
        "mean",
    ),
    bathrooms_median=(
        "bathrooms_clean",
        "median",
    ),
    bathrooms_known_count=(
        "bathrooms_clean",
        "count",
    ),

    elevator_ratio=(
        "elevator_binary",
        "mean",
    ),
    elevator_known_count=(
        "elevator_binary",
        "count",
    ),

    exterior_ratio=(
        "exterior_binary",
        "mean",
    ),
    exterior_known_count=(
        "exterior_binary",
        "count",
    ),
).reset_index()

neighborhood_statistics = (
    neighborhood_statistics.rename(
        columns={
            "zone_key_final": "zone_key",
        }
    )
)

### Añadir cuartiles e intervalos descriptivos

In [8]:
price_m2_quartiles = (
    zone_group["price_per_m2"]
    .quantile(
        [
            0.25,
            0.75,
        ]
    )
    .unstack()
    .rename(
        columns={
            0.25: "price_m2_q1",
            0.75: "price_m2_q3",
        }
    )
    .reset_index()
    .rename(
        columns={
            "zone_key_final": "zone_key",
        }
    )
)

neighborhood_statistics = (
    neighborhood_statistics.merge(
        price_m2_quartiles,
        on="zone_key",
        how="left",
        validate="one_to_one",
    )
)

neighborhood_statistics["price_m2_iqr"] = (
    neighborhood_statistics["price_m2_q3"]
    - neighborhood_statistics["price_m2_q1"]
)

### Calcular la cobertura de cada atributo

In [9]:
known_columns = [
    "rooms_known_count",
    "bathrooms_known_count",
    "elevator_known_count",
    "exterior_known_count",
]

for column in known_columns:
    percentage_column = (
        column.replace(
            "_known_count",
            "_coverage_percentage",
        )
    )

    neighborhood_statistics[
        percentage_column
    ] = (
        neighborhood_statistics[column]
        / neighborhood_statistics["num_listings"]
        * 100
    ).round(2)

### Incorporar los 131 barrios oficiales

In [10]:
neighborhood_real_estate = (
    zones[
        [
            "zone_key",
            "neighborhood_name",
            "district_name",
        ]
    ]
    .merge(
        neighborhood_statistics,
        on="zone_key",
        how="left",
        validate="one_to_one",
    )
    .merge(
        coverage_by_zone[
            [
                "zone_key",
                "total_listings",
                "eligible_listings",
                "eligible_percentage",
                "reliability_level",
            ]
        ],
        on="zone_key",
        how="left",
        validate="one_to_one",
    )
)

count_columns = [
    "num_listings",
    "total_listings",
    "eligible_listings",
    "rooms_known_count",
    "bathrooms_known_count",
    "elevator_known_count",
    "exterior_known_count",
]

for column in count_columns:
    neighborhood_real_estate[column] = (
        neighborhood_real_estate[column]
        .fillna(0)
        .astype(int)
    )

neighborhood_real_estate = (
    neighborhood_real_estate
    .sort_values(
        [
            "district_name",
            "neighborhood_name",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Filas de la tabla inmobiliaria:",
    len(neighborhood_real_estate),
)

print(
    "Barrios con estadísticas:",
    neighborhood_real_estate[
        "num_listings"
    ].gt(0).sum(),
)

display(neighborhood_real_estate.head())

Filas de la tabla inmobiliaria: 131
Barrios con estadísticas: 128


,zone_key,neighborhood_name,district_name,num_listings,price_eur_mean,price_eur_median,price_m2_mean,price_m2_median,price_m2_std,area_m2_mean,...,price_m2_q3,price_m2_iqr,rooms_coverage_percentage,bathrooms_coverage_percentage,elevator_coverage_percentage,exterior_coverage_percentage,total_listings,eligible_listings,eligible_percentage,reliability_level
0,MAD-022,Acacias,Arganzuela,66,400937.863636,399500.0,6121.141800,6020.380435,1362.979403,71.560606,...,7200.000000,1917.417118,78.79,18.18,96.97,96.97,67,66,98.507463,high
1,MAD-027,Atocha,Arganzuela,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,no_data
2,MAD-023,Chopera,Arganzuela,34,420111.470588,399950.0,5168.277818,5165.789474,652.149881,80.970588,...,5528.196347,733.146489,100.00,20.59,100.00,100.00,35,34,97.142857,medium
3,MAD-025,Delicias,Arganzuela,60,399487.966667,313495.0,5458.609409,5319.933555,1024.637516,72.716667,...,5879.248904,1050.240839,95.00,15.00,98.33,95.00,68,60,88.235294,high
4,MAD-021,Imperial,Arganzuela,71,552415.492958,515000.0,5807.415935,5588.235294,1358.433748,96.253521,...,6536.433365,1779.676609,100.00,15.49,98.59,98.59,73,71,97.260274,high


### Validar y guardar

In [11]:
assert len(neighborhood_real_estate) == 131

assert (
    neighborhood_real_estate["zone_key"]
    .duplicated()
    .sum()
    == 0
)

assert (
    neighborhood_real_estate["num_listings"].sum()
    == len(listings_market)
)

coverage_path = (
    PROCESSED_DIR
    / "neighborhood_coverage_audit.csv"
)

summary_path = (
    PROCESSED_DIR
    / "neighborhood_real_estate_summary.csv"
)

coverage_by_zone.to_csv(
    coverage_path,
    index=False,
    encoding="utf-8-sig",
)

neighborhood_real_estate.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

print("Auditoría de cobertura guardada en:")
print(coverage_path)

print("\nResumen inmobiliario guardado en:")
print(summary_path)

print(
    "\nTotal de anuncios agregados:",
    neighborhood_real_estate[
        "num_listings"
    ].sum(),
)

Auditoría de cobertura guardada en:
c:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\processed\neighborhood_coverage_audit.csv

Resumen inmobiliario guardado en:
c:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\processed\neighborhood_real_estate_summary.csv

Total de anuncios agregados: 11058
